# Hafta 7 – Colab çalıştırma defteri (Takım Tertipler)

**Başlamadan önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **GPU** (T4 ya da L4) → Kaydet.

Hücreleri **yukarıdan aşağıya sırayla** çalıştırın (Shift + Enter). Bir hücre kırmızı hata verirse durun,
hatanın son satırlarını Claude'a gönderin.

| Hücre | İş | Süre (T4) |
|---|---|---|
| 1–3 | Hazırlık, kodlar, kütüphaneler | ~3 dk |
| 4 | Veri ve öznitelikleri Drive'dan geri yükleme (yoksa 4b) | ~2 dk (4b: ~30 dk) |
| 5 | wav2vec katman dosyaları (yoksa üretilir) | 0 / ~10 dk |
| 6 | Klasik modeller (yoksa) | ~5 dk |
| 7 | CNN ve CNN-LSTM, tek bölümleme | ~15 dk |
| 8 | CNN ve CNN-LSTM, 5 katlı CV | ~70 dk |
| 9 | wav2vec katman birleşimi deneyleri | ~25 dk |
| 10 | Tüm modeller aynı katlarda: McNemar / t-testi | ~5 dk |
| 11 | **Sonuçları indir + modelleri Drive'a yedekle** | ~1 dk |
| 12 | Arayüz kabul testi | ~5 dk |

Uzun adımlar sürerken sekmeyi açık tutun. İki oturuma bölmek isterseniz her oturumun sonunda **Hücre 11**'i çalıştırın.

In [ ]:
# Hücre 1 – GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

In [ ]:
# Hücre 2 – Kodlar
# Hafta 7 kodları GitHub'a yüklendiyse depodan çekilir; yüklenmediyse hafta7_depoya_cikar.zip dosyasını yüklemeniz istenir.
import os, subprocess
%cd /content
if not os.path.isdir('/content/proje'):
    subprocess.run('git clone -q https://github.com/DuhannK/konusmadan-duygu-tanima.git proje', shell=True)
%cd /content/proje
if not os.path.exists('src/w2v_experiments.py'):
    print("Depoda Hafta 7 kodları yok -> hafta7_depoya_cikar.zip dosyasını seçin")
    from google.colab import files
    up = files.upload()
    for name in up:
        !unzip -o -q "{name}" -d /content/proje
assert os.path.exists('src/w2v_experiments.py'), "Hafta 7 kodları bulunamadı"
!mkdir -p logs
!ls src

In [ ]:
# Hücre 3 – Kütüphaneler
!pip install -q opensmile transformers streamlit
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# Hücre 4 – Veri ve öznitelikleri Drive'dan geri yükle
# Drive izin penceresinde yalnızca "Google Drive dosyalarını görme, düzenleme..." izni yeterli.
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil
%cd /content/proje
MY = '/content/drive/MyDrive'
# 1) Hafta 4'te önerilen yedek: MyDrive/ser/veri_v3.zip (ya da benzer adlı bir zip)
zips = sorted(glob.glob(f'{MY}/ser/*.zip') + glob.glob(f'{MY}/**/veri_v3*.zip', recursive=False))
for z in zips:
    print("zip bulundu:", z); os.system(f'unzip -o -q "{z}" -d /content/proje')
# 2) Hafta 6 kılavuzundaki klasör yedekleri: MyDrive/tertipler_*
for src, dst in [('tertipler_processed_v3', 'processed_v3'), ('tertipler_features_v3', 'features_v3'),
                 ('tertipler_results_v3', 'results_v3'), ('tertipler_results_w2v', 'results_w2v')]:
    if os.path.isdir(f'{MY}/{src}'):
        shutil.copytree(f'{MY}/{src}', dst, dirs_exist_ok=True); print("kopyalandı:", dst)
for d in ['processed_v3', 'features_v3', 'results_v3']:
    print(f"{d:14s}", "VAR" if os.path.isdir(d) and os.listdir(d) else "YOK")
print("\nprocessed_v3 ve features_v3 VAR ise Hücre 4b'yi ATLAYIN.")

In [ ]:
# Hücre 4b – YALNIZCA processed_v3 / features_v3 YOKSA: veri indirme + ön işleme + öznitelik (~30 dk)
import os
if os.path.isdir('processed_v3') and os.path.isdir('features_v3'):
    print("Veri zaten var, bu hücre atlandı.")
else:
    import kagglehub
    from pathlib import Path
    rav_root = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
    cre_root = kagglehub.dataset_download("ejlok1/cremad")
    cands = {p.parent.parent for p in Path(rav_root).rglob("03-01-*.wav")}
    RAV = str(max(cands, key=lambda p: len(p.parts)))
    CREMA = str(next(Path(cre_root).rglob("AudioWAV")))
    !wget -q https://raw.githubusercontent.com/CheyneyComputerScience/CREMA-D/master/VideoDemographics.csv
    !python -u src/preprocess.py --ravdess_dir "{RAV}" --cremad_dir "{CREMA}" --demographics VideoDemographics.csv --out_dir processed_v3 --augment 2>&1 | tail -3
    !python -u src/features.py --in_dir processed_v3 --out_dir features_v3 --skip_logmel 2>&1 | tail -3

In [ ]:
# Hücre 5 – wav2vec katman dosyaları (w2v_layers_*.npy). Yoksa üretilir (~10 dk, GPU).
import os
if all(os.path.exists(f'features_v3/w2v_layers_{s}.npy') for s in ['train', 'val', 'test']):
    print("w2v_layers dosyaları VAR.")
else:
    !python -u src/extra_features.py --in_dir processed_v3 --out_dir features_v3 --sets w2v --batch 64 2>&1 | tee logs/h7_0_w2v_features.txt | grep -v "^   w2v"
# Sonraki oturumlar için veriyi (w2v katmanları dahil) Drive'a yedekle
!mkdir -p /content/drive/MyDrive/ser
!zip -q -r -0 /content/drive/MyDrive/ser/veri_v3.zip processed_v3 features_v3 && echo "Drive yedeği güncellendi"

In [ ]:
# Hücre 6 – Klasik modeller (CV parametreleri ve arayüzün klasik seçeneği için). Varsa atlanır.
import os
if os.path.exists('results_v3/results.json') and os.path.exists('results_v3/model_svm.joblib'):
    print("results_v3 VAR.")
else:
    !python -u src/baseline.py --feat_dir features_v3 --out_dir results_v3 2>&1 | tee logs/h7_1_baseline.txt | tail -9

In [ ]:
# Hücre 7 – Düzeltilmiş CNN ve CNN-LSTM, birleşik veri, tek bölümleme (40 epok, en iyi doğrulama epoğu)
!python -u src/cnn.py --in_dir processed_v3 --dataset all --model cnn      --out_dir cnn_all      2>&1 | tee logs/h7_2_cnn_all.txt      | grep -v "^epok"
!python -u src/cnn.py --in_dir processed_v3 --dataset all --model cnn_lstm --out_dir cnn_lstm_all 2>&1 | tee logs/h7_3_cnn_lstm_all.txt | grep -v "^epok"

In [ ]:
# Hücre 8 – CNN ve CNN-LSTM, konuşmacı bazlı 5 katlı CV (birleşik veri) – uzun sürer
!python -u src/cnn.py --in_dir processed_v3 --dataset all --model cnn      --cv 5 --out_dir cnn_cv_all      2>&1 | tee logs/h7_4_cnn_cv_all.txt      | grep -E "TEST|CV:"
!python -u src/cnn.py --in_dir processed_v3 --dataset all --model cnn_lstm --cv 5 --out_dir cnn_lstm_cv_all 2>&1 | tee logs/h7_5_cnn_lstm_cv_all.txt | grep -E "TEST|CV:"

In [ ]:
# Hücre 9 – wav2vec 2.0 katman birleşimi: tek katman / tüm katmanlar / orta katmanlar / en iyi 3 (+ 5 katlı CV)
!python -u src/w2v_experiments.py --in_dir processed_v3 --feat_dir features_v3 --dataset all --cv_all --out_dir w2v_exp_all 2>&1 | tee logs/h7_6_w2v_exp.txt

In [ ]:
# Hücre 10 – Tüm modeller aynı katlarda: SVM (556 öznitelik) + CNN + CNN-LSTM + wav2vec stratejileri
!python -u src/cv_eval.py --in_dir processed_v3 --feat_dir features_v3 --dataset all --params results_v3/results.json --models svm --include_dir cnn_cv_all cnn_lstm_cv_all w2v_exp_all --out_dir cv_all_hafta7 2>&1 | tee logs/h7_7_cv_all.txt | tail -60

In [ ]:
# Hücre 11 – Sonuçları indir (Claude'a gönderilecek) + model dosyalarını arayüz için Drive'a yedekle
from google.colab import files
!zip -q -r hafta7_sonuclar.zip logs cnn_all cnn_lstm_all cnn_cv_all cnn_lstm_cv_all w2v_exp_all cv_all_hafta7 -x "*.pt" -x "*.joblib" -x "*.npy"
!mkdir -p /content/drive/MyDrive/ser
!zip -q -r /content/drive/MyDrive/ser/modeller_hafta7.zip cnn_all/*.pt cnn_lstm_all/*.pt w2v_exp_all/model results_v3/*.joblib && echo "modeller Drive'a yedeklendi"
files.download("hafta7_sonuclar.zip")

## Hücre 12 – Arayüz kabul testi (H6)

Aşağıdaki hücre arayüzü en iyi wav2vec modeli, CNN ve klasik modelle açar ve bir bağlantı yazdırır. Bağlantıyı açıp
şu kontrolleri yapın; her birinin sonucunu (çalıştı / hata) not edin:

1. Soldaki listede **üç model** de görünüyor mu?
2. **Dosya yükleme:** RAVDESS'ten öfkeli bir kayıt (dosya adının 3. alanı `05`) yükleyin. Tahmin ve olasılık çubukları geliyor mu?
3. **Uzun kayıt:** 5 sn'den uzun bir kayıt yükleyin. "pencerenin ortalaması" notu çıkıyor mu?
4. **Mikrofon:** 2–4 sn konuşun (tarayıcı izin isterse verin). Tahmin geliyor mu?
5. **Sessiz kayıt:** Hiç konuşmadan kısa bir kayıt alın. "konuşma bulunamadı" uyarısı çıkıyor mu, uygulama çökmüyor mu?
6. Her üç modelle aynı dosyayı deneyin; tahmin süresi kaç saniye?
7. Bir ekran görüntüsü alın (rapor için).

In [ ]:
# Hücre 12 – Arayüzü başlat ve bağlantıyı yazdır
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess, time, re
subprocess.Popen("streamlit run src/app.py --server.port 8501 --server.headless true -- "
                 "--w2v_dir w2v_exp_all/model --feat_info w2v_exp_all/model/w2v_info.json "
                 "--cnn cnn_all/cnn_best.pt --classic_dir results_v3 > streamlit.log 2>&1", shell=True)
time.sleep(8)
p = subprocess.Popen("cloudflared tunnel --url http://localhost:8501 --no-autoupdate",
                     shell=True, stderr=subprocess.PIPE, text=True)
for line in p.stderr:
    m = re.search(r"https://[-\w.]+trycloudflare\.com", line)
    if m:
        print("\nARAYÜZ BAĞLANTISI:", m.group(0), "\n"); break

In [ ]:
# Arayüz hata verirse: son satırları Claude'a gönderin
!tail -30 streamlit.log